### 测试 jax.lax.reduce_sum

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import jax
import jax.numpy as jnp
import torch
import numpy as np
jax.config.update("jax_default_matmul_precision", "highest")
jax.config.update("jax_enable_x64", True)
rng = np.random.default_rng(seed=0)
print(jax.devices())

[CudaDevice(id=0)]


In [2]:
x = rng.normal(size=(4096,))

In [3]:
def to_numpy_f32(arr):
    if isinstance(arr, torch.Tensor):
        arr = arr.detach().cpu().numpy()
    else:
        arr = np.asarray(arr)
    return np.ascontiguousarray(arr.astype(np.float32, copy=False))
def ulp_diff_float32(a, b):
    a_i = a.view(np.int32).astype(np.int64)
    b_i = b.view(np.int32).astype(np.int64)
    a_ordered = np.where(a_i < 0, 0x80000000 - a_i, a_i)
    b_ordered = np.where(b_i < 0, 0x80000000 - b_i, b_i)
    return np.abs(a_ordered - b_ordered)
def compare_metrics(name, test, ref):
    test = to_numpy_f32(test)
    ref = to_numpy_f32(ref)

    finite_mask = np.isfinite(test) & np.isfinite(ref)
    skipped = test.size - int(finite_mask.sum())
    test_f = test[finite_mask]
    ref_f = ref[finite_mask]

    abs_err = np.abs(test_f - ref_f)
    rel_denom = np.maximum(np.abs(ref_f), np.finfo(np.float32).eps)
    rel_err = abs_err / rel_denom
    ulp_err = ulp_diff_float32(test_f, ref_f)

    print(f"\n{name}")
    print(
        f"  valid elements: {test_f.size}/{test.size}, skipped non-finite pairs: {skipped}"
    )
    print(
        f"  abs_err: max={abs_err.max():.6e}, mean={abs_err.mean():.6e}, p99={np.percentile(abs_err, 99):.6e}"
    )
    print(
        f"  rel_err: max={rel_err.max():.6e}, mean={rel_err.mean():.6e}, p99={np.percentile(rel_err, 99):.6e}"
    )
    print(
        f"  ulp_err: max={int(ulp_err.max())}, mean={ulp_err.mean():.6f}, p99={np.percentile(ulp_err, 99):.6f}"
    )
def find_max_error_index(o_jax, o_torch):
    abs_diff = np.abs(to_numpy_f32(o_jax) - to_numpy_f32(o_torch))

    # 找到全局误差最大的索引位置
    max_diff_idx = np.unravel_index(np.argmax(abs_diff), abs_diff.shape)

    print(f"最大误差发生处的索引: {max_diff_idx}")
    print(f"JAX输入 x: {x[max_diff_idx]}")
    print(f"JAX输出: {o_jax[max_diff_idx]}")
    print(f"Torch输出: {o_torch[max_diff_idx]}")

In [4]:
x_jax_16 = jnp.array(x,dtype=jnp.bfloat16)
x_torch_16 = torch.from_numpy(x).to(torch.bfloat16).cuda()

In [5]:
o_jax_16 = jax.lax.reduce_sum(x_jax_16,axes=(0,)).astype(jnp.float32)
o_torch_16 = torch.ops.aten.sum.default(x_torch_16).to(torch.float32)
compare_metrics("bf16: jax vs torch", o_jax_16, o_torch_16)


bf16: jax vs torch
  valid elements: 1/1, skipped non-finite pairs: 0
  abs_err: max=1.000000e+00, mean=1.000000e+00, p99=1.000000e+00
  rel_err: max=1.503759e-02, mean=1.503759e-02, p99=1.503759e-02
  ulp_err: max=131072, mean=131072.000000, p99=131072.000000


In [6]:
o_jax_16

Array(-65.5, dtype=float32)

In [7]:
o_torch_16

tensor(-66.5000, device='cuda:0')

In [8]:
x_jax_32 = jnp.array(x,dtype=jnp.float32)
x_torch_32 = torch.from_numpy(x).to(torch.float32).cuda()
o_jax_32 = jax.lax.reduce_sum(x_jax_32,axes=(0,)).astype(jnp.float32)
o_torch_32 = torch.ops.aten.sum.default(x_torch_32).to(torch.float32)
compare_metrics("fp32: jax vs torch", o_jax_32, o_torch_32)


fp32: jax vs torch
  valid elements: 1/1, skipped non-finite pairs: 0
  abs_err: max=0.000000e+00, mean=0.000000e+00, p99=0.000000e+00
  rel_err: max=0.000000e+00, mean=0.000000e+00, p99=0.000000e+00
  ulp_err: max=0, mean=0.000000, p99=0.000000


In [9]:
o_torch_16 = torch.ops.aten.sum.default(x_torch_16).to(torch.float32)
o_torch_32 = torch.ops.aten.sum.default(x_torch_32).to(torch.float32)
x_torch_32_test = x_torch_16.to(torch.float32)
o_torch_32_test = torch.ops.aten.sum.default(x_torch_32_test).to(torch.float32)
print(f"o_torch_16: {o_torch_16}")
print(f"o_torch_32: {o_torch_32}")
print(f"o_torch_32_test: {o_torch_32_test}")

o_torch_16: -66.5
o_torch_32: -66.06460571289062
o_torch_32_test: -66.28488159179688


In [13]:
# 告诉底层 C++ Kernel：给我用 fp32 累加，并且直接返回 fp32 张量
o_torch_16_safe = torch.ops.aten.sum.default(x_torch_16, dtype=torch.float32).to(torch.float32)
print(f"o_torch_16_safe: {o_torch_16_safe}")

o_torch_16_safe: -66.2848892211914


In [14]:
@torch.compile
def fn(x):
    return torch.ops.aten.sum.default(x).to(torch.float32)

print(fn(x_torch_32_test))

tensor(-66.2849, device='cuda:0')


In [12]:
def fn(x):
    return torch.ops.aten.sum.default(x)

fn(x_torch_32_test).to(torch.float32)

tensor(-66.2849, device='cuda:0')